# Tutorial 4: Visualization Gallery

This tutorial demonstrates how to create comprehensive visualizations of pressing patterns. You'll learn:

- How to create pressing heatmaps showing spatial density
- How to visualize co-pressing networks
- How to plot player trajectories
- How to use the Streamlit app for interactive animations
- How to create custom pitch diagrams

## Prerequisites

Complete Tutorials 1-3 to have:
- Extracted build-ups
- Computed features
- Trained models

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from scipy.spatial import ConvexHull
import networkx as nx

# Visualization modules
from src.viz.pressing_heatmap import create_pressing_heatmap
from src.viz.network import build_pressing_network, plot_pressing_network
from src.viz.plots import draw_pitch
from src.features.services.window_loader import WindowLoader
from src.features.services.normalization import normalize_coordinates
from src.features.services.possession import infer_ball_carrier
from src.features.services.utils import prepare_frame_data, time_to_seconds
from src.features.services.metadata import enrich_with_team_id

# Configuration
PROCESSED_ROOT = Path("data/processed/rm_pressing_tutorial")
OUTPUT_DIR = Path("visualizations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("white")
plt.rcParams['figure.dpi'] = 100

## Part 1: Pressing Heatmaps

### Step 1: Create Single Build-Up Heatmap

Visualize where Real Madrid players position themselves during a build-up.

In [ ]:
loader = WindowLoader(PROCESSED_ROOT)
index = loader.index

# Load first build-up
build_up_id = index.iloc[0]['build_up_id']
df = loader.load_build_up(build_up_id)
meta = loader.get_metadata(build_up_id)

# Preprocess
df = prepare_frame_data(df)
df = enrich_with_team_id(df, meta['game_id'])
df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))

# Create heatmap
fig = create_pressing_heatmap(
    df_norm,
    meta.get('rm_team_id'),
    title=f"Real Madrid Pressing Heatmap - Build-up {build_up_id}"
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"heatmap_build_up_{build_up_id}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to {OUTPUT_DIR / f'heatmap_build_up_{build_up_id}.png'}")

### Step 2: Aggregate Heatmap Across Multiple Build-Ups

Combine positions from many build-ups to show overall pressing tendencies.

In [ ]:
from tqdm.notebook import tqdm

# Collect all RM player positions across first 20 build-ups
all_positions = []

for bid in tqdm(index['build_up_id'].tolist()[:20], desc="Loading build-ups"):
    try:
        df = loader.load_build_up(bid)
        meta = loader.get_metadata(bid)
        
        df = prepare_frame_data(df)
        df = enrich_with_team_id(df, meta['game_id'])
        df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
        
        # Get RM positions
        rm_df = df_norm[df_norm['team_id'] == meta.get('rm_team_id')]
        all_positions.extend(rm_df[['x_norm', 'y_norm']].values.tolist())
    except Exception:
        continue

all_positions_df = pd.DataFrame(all_positions, columns=['x_norm', 'y_norm'])

print(f"Collected {len(all_positions_df)} positions from {20} build-ups")

# Create aggregate heatmap using manual binning
fig, ax = plt.subplots(figsize=(12, 8))

# Pitch outline
draw_pitch(ax)

# 2D histogram heatmap
x_bins = np.linspace(-52.5, 52.5, 22)  # 21 bins
y_bins = np.linspace(-34, 34, 14)  # 13 bins

heatmap, x_edges, y_edges = np.histogram2d(
    all_positions_df['x_norm'],
    all_positions_df['y_norm'],
    bins=[x_bins, y_bins]
)

# Plot heatmap
im = ax.imshow(
    heatmap.T,
    extent=[-52.5, 52.5, -34, 34],
    origin='lower',
    cmap='hot',
    alpha=0.7,
    aspect='auto'
)

plt.colorbar(im, ax=ax, label='Position Density', fraction=0.046, pad=0.04)
ax.set_title('Aggregate Real Madrid Pressing Heatmap (20 Build-Ups)', fontsize=14, fontweight='bold')
ax.set_xlabel('X (meters)', fontsize=12)
ax.set_ylabel('Y (meters)', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "aggregate_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Player Trajectories

### Step 3: Visualize Individual Player Movement

Plot the path a pressing player takes during a build-up.

In [ ]:
# Reload first build-up with time info
df = loader.load_build_up(build_up_id)
meta = loader.get_metadata(build_up_id)

df = prepare_frame_data(df)
df = enrich_with_team_id(df, meta['game_id'])
df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)

# Get RM players
rm_df = df_norm[df_norm['team_id'] == meta.get('rm_team_id')]

# Pick a player who appears in most frames
player_frame_counts = rm_df.groupby('player_id').size().sort_values(ascending=False)
top_player = player_frame_counts.index[0]

player_trajectory = rm_df[rm_df['player_id'] == top_player].sort_values('time_seconds')

# Plot trajectory
fig, ax = plt.subplots(figsize=(12, 8))
draw_pitch(ax)

# Plot path
ax.plot(player_trajectory['x_norm'], player_trajectory['y_norm'], 
        color='blue', linewidth=3, alpha=0.7, label=f'Player {top_player}')

# Mark start and end
ax.scatter(player_trajectory.iloc[0]['x_norm'], player_trajectory.iloc[0]['y_norm'],
          s=200, color='green', marker='o', edgecolor='black', linewidth=2, 
          label='Start', zorder=10)
ax.scatter(player_trajectory.iloc[-1]['x_norm'], player_trajectory.iloc[-1]['y_norm'],
          s=200, color='red', marker='X', edgecolor='black', linewidth=2, 
          label='End', zorder=10)

# Add time markers every 1 second
for i, row in player_trajectory.iterrows():
    if int(row['time_seconds']) % 1 == 0:  # Every second
        ax.text(row['x_norm'], row['y_norm'] + 2, f"{int(row['time_seconds'])}s",
               fontsize=9, ha='center', color='darkblue', fontweight='bold')

ax.set_title(f"Player {top_player} Trajectory - Build-up {build_up_id}", 
            fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"trajectory_player_{top_player}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Player {top_player} moved {len(player_trajectory)} frames")
total_distance = np.sum(np.sqrt(np.diff(player_trajectory['x_norm'])**2 + 
                                np.diff(player_trajectory['y_norm'])**2))
print(f"Total distance covered: {total_distance:.1f} meters")

## Part 3: Team Shape Visualization

### Step 4: Convex Hull at Different Time Points

Show how Real Madrid's shape changes from kick to engagement.

In [ ]:
kick_time = time_to_seconds(str(meta.get('kick_time')))
time_points = [kick_time + 1, kick_time + 3, kick_time + 5]
labels = ['Kick + 1s', 'Kick + 3s', 'Kick + 5s']
colors = ['blue', 'purple', 'red']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (t, label, color) in enumerate(zip(time_points, labels, colors)):
    ax = axes[i]
    draw_pitch(ax)
    
    # Find closest frame to time point
    frame_times = df_norm[['frame', 'time_seconds']].drop_duplicates()
    closest_frame_idx = (frame_times['time_seconds'] - t).abs().idxmin()
    closest_frame = frame_times.loc[closest_frame_idx, 'frame']
    
    # Get RM positions at this frame
    frame_df = df_norm[(df_norm['frame'] == closest_frame) & 
                      (df_norm['team_id'] == meta.get('rm_team_id'))]
    
    if len(frame_df) < 3:
        ax.set_title(f"{label} - Insufficient players", fontsize=12)
        continue
    
    positions = frame_df[['x_norm', 'y_norm']].values
    
    # Plot players
    ax.scatter(positions[:, 0], positions[:, 1], s=100, color=color, 
              edgecolor='black', linewidth=1.5, alpha=0.8, zorder=5)
    
    # Compute and plot convex hull
    try:
        hull = ConvexHull(positions)
        for simplex in hull.simplices:
            ax.plot(positions[simplex, 0], positions[simplex, 1], 
                   color=color, linewidth=2, alpha=0.7)
        
        # Fill hull
        hull_points = positions[hull.vertices]
        ax.fill(hull_points[:, 0], hull_points[:, 1], 
               color=color, alpha=0.2)
        
        area = hull.volume  # In 2D, volume = area
        ax.text(0, -38, f"Area: {area:.0f} m²", ha='center', fontsize=11,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    except Exception:
        pass
    
    ax.set_title(label, fontsize=12, fontweight='bold')

plt.suptitle(f"Real Madrid Shape Evolution - Build-up {build_up_id}", 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shape_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

## Part 4: Co-Pressing Networks

### Step 5: Build Pressing Partnership Network

Identify which players press together frequently.

In [ ]:
from src.models.pressing_affinity import compute_pressing_affinity
from src.models.gmm_zones import identify_pressers
from src.models.config import GMMConfig

# Collect presser sets across build-ups
presser_sets = []

gmm_config = GMMConfig()

for bid in tqdm(index['build_up_id'].tolist()[:30], desc="Identifying pressers"):
    try:
        df = loader.load_build_up(bid)
        meta = loader.get_metadata(bid)
        
        df = prepare_frame_data(df)
        df = enrich_with_team_id(df, meta['game_id'])
        df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
        df_norm = infer_ball_carrier(df_norm, meta.get('opponent_team_id'))
        df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)
        
        pressers = identify_pressers(df_norm, gmm_config)
        if len(pressers) > 0:
            presser_sets.append(set(pressers))
    except Exception:
        continue

print(f"Collected {len(presser_sets)} presser sets")

# Compute affinity matrix
affinity_df = compute_pressing_affinity(presser_sets, metric='jaccard')

print(f"\nAffinity matrix: {affinity_df.shape}")
print(affinity_df.head())

### Step 6: Visualize Network Graph

Plot pressing partnerships as a network.

In [ ]:
# Build network graph
G = build_pressing_network(affinity_df, threshold=0.3)

print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Plot network
fig = plot_pressing_network(
    G,
    title="Real Madrid Co-Pressing Network (First 30 Build-Ups)",
    node_size=800,
    edge_width_scale=3
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pressing_network.png", dpi=150, bbox_inches='tight')
plt.show()

# Network statistics
if G.number_of_edges() > 0:
    print("\nTop 5 Pressing Partnerships (by edge weight):")
    edges = [(u, v, G[u][v]['weight']) for u, v in G.edges()]
    edges_sorted = sorted(edges, key=lambda x: x[2], reverse=True)[:5]
    for u, v, w in edges_sorted:
        print(f"  Player {u} ↔ Player {v}: {w:.3f}")

## Part 5: Feature Visualization

### Step 7: Feature Scatter Plot with Topic Coloring

Visualize build-ups in feature space, colored by dominant topic.

In [ ]:
# Load features
features_df = pd.read_parquet(PROCESSED_ROOT / "features.parquet")

# Load topic assignments (if available)
topics_file = Path("data/processed/rm_pressing_tutorial_topics/build_up_topic_weights.parquet")
if topics_file.exists():
    H_df = pd.read_parquet(topics_file)
    dominant_topics = H_df.idxmax(axis=1)
    
    # Merge with features
    features_df = features_df.merge(
        dominant_topics.rename('dominant_topic').to_frame(),
        left_on='build_up_id',
        right_index=True,
        how='left'
    )
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for topic in features_df['dominant_topic'].unique():
        if pd.notna(topic):
            subset = features_df[features_df['dominant_topic'] == topic]
            ax.scatter(
                subset['t_first_pressure_s'],
                subset['rm_width_mean_m'],
                label=topic,
                s=80,
                alpha=0.7,
                edgecolor='black',
                linewidth=0.5
            )
    
    ax.set_xlabel('Time to First Pressure (s)', fontsize=12)
    ax.set_ylabel('Team Width (m)', fontsize=12)
    ax.set_title('Build-Ups in Feature Space (Colored by Dominant Topic)', 
                fontsize=14, fontweight='bold')
    ax.legend(title='Dominant Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "features_by_topic.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Topic weights not found. Run Tutorial 3 first.")

## Part 6: Using the Streamlit App

### Step 8: Launch Interactive Viewer

The Streamlit app provides an interactive interface for exploring build-ups.

In [ ]:
print("To launch the Streamlit app, run this command in your terminal:\n")
print("  streamlit run app.py\n")
print("Features:")
print("  - 🎬 Viewer Tab: Animated playback with statistical overlays")
print("  - 📊 Dashboard Tab: Aggregate analytics across all build-ups")
print("  - 🔀 Comparison Tab: Side-by-side analysis of multiple build-ups")
print("\nControls:")
print("  - Select build-up from dropdown")
print("  - Choose phase (Full window, Ready → Kick, Kick → End)")
print("  - Enable overlays (Timeline, Convex Hull, Voronoi, Vectors)")
print("  - Adjust min players threshold")
print("  - Use frame navigator for precise control")

## Summary

You've learned how to create:
1. ✅ Pressing heatmaps (single and aggregate)
2. ✅ Player trajectory plots
3. ✅ Convex hull shape evolution diagrams
4. ✅ Co-pressing network graphs
5. ✅ Feature space scatter plots with topic coloring
6. ✅ Interactive Streamlit visualizations

## Visualization Gallery Summary

All visualizations saved to `visualizations/`:
- `heatmap_build_up_*.png`: Single build-up heatmaps
- `aggregate_heatmap.png`: Combined spatial density
- `trajectory_player_*.png`: Individual player movements
- `shape_evolution.png`: Team compactness over time
- `pressing_network.png`: Co-pressing partnerships
- `features_by_topic.png`: Build-ups clustered by pressing pattern

## Next Steps

- Combine multiple visualization types in reports
- Create video animations of pressing sequences
- Export to presentation-ready formats
- Build custom dashboards for match analysis